# Geometric Deep Learning on BIM: From IFC to Graph Neural Networks

**Author:** Olga Poletkina  
**Course:** ICD/CA — Industry Connected Digitalization / Computational Architecture  
**Dataset:** Duplex.ifc (buildingSMART sample model)  

---

## Notebook Overview

This notebook presents an end-to-end pipeline for applying **geometric deep learning** to a Building Information Model starting from a raw IFC file.

Step by step, we:

1. Extract geometric and semantic data from IFC elements (walls, doors, windows, spaces).
2. Explore alternative geometric representations (mesh, point cloud, graph) and discuss their pros and cons.
3. Build a **graph** with typed edges based on IFC relations (and, where needed, represent it as `HeteroData`).
4. Engineer node and edge features so they can be consumed by a GNN.
5. Train a relation-aware GNN (R-GCN) on a BIM-related task.
6. Validate predictions using **SHACL constraints** as an explainability and compliance layer.

Most of the data extraction and graph construction code is reused from my master’s thesis pipeline and adapted to the Duplex.ifc example.

The overall workflow is framed as an **architectural co-pilot**: the system does not replace the designer, but helps them by highlighting anomalies and potential constraint violations in the model.

---

## 1. Co-Pilot Framing: Motivation & Application Context

### 1.1 Problem Statement

Modern BIM models contain hundreds or thousands of interconnected elements. Each element has geometry and a set of parameters. Manually checking such models (clashes, code compliance, design consistency) is slow and expensive.

The goal is to build a **GNN-based co-pilot** that:

- reads a BIM model as a graph,
- learns typical structural and geometric patterns between elements,
- flags anomalies and missing/suspicious properties,
- and links its predictions to formal rules (SHACL) so they can be explained and audited.

### 1.2 End-to-End Workflow

The high-level workflow is:

```
Architect uploads IFC
       │
       ▼
┌─────────────────────┐
│ Geometry Extraction │  ← mesh generation, centroids, extents, orientations
└────────┬────────────┘
         ▼
┌─────────────────────┐
│ Graph Construction  │  ← IFC relations (SpaceBoundary, Fills, Voids) → typed edges
└────────┬────────────┘
         ▼
┌─────────────────────┐
│ Feature Engineering │  ← geometric + semantic features per node / edge
└────────┬────────────┘
         ▼
┌─────────────────────┐
│ GNN Prediction      │  ← R-GCN on a homogeneous graph (from the master’s pipeline) or HeteroGNN
└────────┬────────────┘
         ▼
┌─────────────────────┐
│ SHACL Validation    │  ← formal constraint checking on model + predictions
└────────┬────────────┘
         ▼
Co-pilot presents findings → Architect reviews & acts
```

In this notebook, each stage is backed by code from my master’s thesis pipeline (IFC extraction, graph construction, feature engineering, R-GCN), plus additional blocks for SHACL and visualisation to show how everything fits into a single coherent framework.
```

---

## 2. Setup & Dependencies

In [1]:
import sys, os, json, warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt

import ifcopenshell
import ifcopenshell.geom
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=FutureWarning)

MLOPS_ROOT = Path(r"C:\Users\olgap\OneDrive\MIPT_Master\PIK_MIPT_anomaly_detection\mlops_demo")
sys.path.insert(0, str(MLOPS_ROOT))

from src.extract.data_extractor import (
    IFCMeshGeometryDataExtractor,
    _iter_element_psets,
    _ifc_value_to_py,
)
from src.extract.graph_create import (
    build_ifc_graph,
    validate_ifc_graph,
    clean_graph_structure,
)
from src.utils.graph_create import (
    plot_ifc_graph_3d,
    separate_disconnected_components,
)

IFC_PATH = Path(r"C:\Users\olgap\OneDrive\MIPT_Master\PIK_MIPT_anomaly_detection\ICD_CA_assignment\data\Duplex.ifc")

print(f"ifcopenshell {ifcopenshell.version}")
print(f"IFC file {IFC_PATH.name}  ({IFC_PATH.stat().st_size / 1e6:.1f} MB)")
print("Imports OK")

ifcopenshell 0.8.4.post1
IFC file Duplex.ifc  (2.4 MB)
Imports OK


---

## 3. IFC Data Extraction

### 3.1 Loading the IFC Model


In [2]:
ifc_file = ifcopenshell.open(str(IFC_PATH))

print(f"Schema: {ifc_file.schema}")
print(f"Header: {ifc_file.header.file_name.name}")

elements = ifc_file.by_type("IfcElement")
spaces   = ifc_file.by_type("IfcSpace")

type_counts = Counter(e.is_a() for e in list(elements) + list(spaces))
type_df = (
    pd.DataFrame(type_counts.items(), columns=["IFC Type", "Count"])
    .sort_values("Count", ascending=False)
    .reset_index(drop=True)
)

print(f"\nTotal elements (incl. spaces): {sum(type_counts.values())}")
print(f"Distinct IFC types: {len(type_counts)}\n")
type_df

Schema: IFC2X3
Header: 0001

Total elements (incl. spaces): 289
Distinct IFC types: 16



,IFC Type,Count
0,IfcFurnishingElement,61
1,IfcWallStandardCase,56
2,IfcOpeningElement,50
3,IfcWindow,24
4,IfcSlab,21
5,IfcSpace,21
6,IfcDoor,14
7,IfcCovering,13
8,IfcBeam,8
9,IfcFooting,7


### 3.2 Geometry Extraction via Mesh Tessellation

In IFC, element geometry is stored in a parametric form (B-Rep: extrusions, booleans, etc.). To use it in a GNN, we first need explicit 3D points. For this, we **tessellate** each element into a triangle mesh using `ifcopenshell.geom`.

From this mesh we then compute an **Oriented Bounding Box (OBB)** via PCA. This gives a compact description of each element’s size, position, and orientation.

| Feature            | Description                                   | Use in the graph                  |
|--------------------|-----------------------------------------------|-----------------------------------|
| `centroid_x/y/z`   | Centre of mass of mesh vertices               | Node position (`pos`)             |
| `obb_extent_x/y/z` | Half-lengths along 3 principal PCA axes       | Length / width / height           |
| `angle_to_x/y`     | Angle of main PCA axis to global X/Y (degrees)| Orientation feature (e.g. walls)  |
| `vertices`         | Raw (x, y, z) mesh vertices                   | For point cloud features (Sec. 4) |

**Why OBB instead of AABB?**  
Many BIM elements (walls, beams) are rotated with respect to the global axes. An axis-aligned bounding box (AABB) “inflates” their size and does not directly give the true length and thickness. The OBB is aligned to the element’s principal axes, so its extents correspond to meaningful dimensions. If needed, an AABB can always be recovered from the OBB, so we do not store redundant AABB columns.

**Tessellation resolution.**  
Finer tessellation gives more accurate centroids and smoother point clouds, but increases memory and runtime. For typical BIM elements (planar walls, rectangular doors), a moderate/coarse tessellation is usually sufficient.


In [3]:
N_PC_POINTS = 256
np.random.seed(42)

_ifc_pc = ifcopenshell.open(str(IFC_PATH))          
_settings = ifcopenshell.geom.settings()
_settings.set(_settings.USE_WORLD_COORDS, True)

_all_el = list(_ifc_pc.by_type("IfcElement")) + list(_ifc_pc.by_type("IfcSpace"))

pc_raw: dict[int, np.ndarray] = {}
_n_fail = 0
_first_err = None

for _el in tqdm(_all_el, desc="Sampling point clouds"):
    try:
        _shape = ifcopenshell.geom.create_shape(_settings, _el)
        _verts = _shape.geometry.verts
        if _verts and len(_verts) >= 9:
            _pts = np.array(_verts).reshape(-1, 3)
            _pts = _pts - _pts.mean(axis=0)
            _idx = np.random.choice(len(_pts), N_PC_POINTS,
                                    replace=len(_pts) < N_PC_POINTS)
            pc_raw[_el.id()] = _pts[_idx]
    except Exception as _e:
        _n_fail += 1
        if _first_err is None:
            _first_err = _e

print(f"Point clouds — ok: {len(pc_raw)} | fail: {_n_fail}")
if _first_err:
    print(f"  first error: {type(_first_err).__name__}: {_first_err}")

del _ifc_pc, _settings, _all_el, _first_err, _n_fail

ifc_file = ifcopenshell.open(str(IFC_PATH))

Sampling point clouds:   0%|          | 0/289 [00:00<?, ?it/s]

Point clouds — ok: 286 | fail: 3
  first error: RuntimeError: Representation is NULL


In [4]:
extractor = IFCMeshGeometryDataExtractor(
    ifc_folder=str(IFC_PATH.parent),
    output_folder=str(IFC_PATH.parent / "output"),
)

nodes_df = extractor.extract_nodes_df(ifc_file)
edges_df_raw = extractor.extract_edges(ifc_file)

nodes_df = extractor._add_wall_long_axis(nodes_df)
nodes_df = extractor._add_host_wall_id(nodes_df, edges_df_raw)

drop_cols = ["size_x", "size_y", "size_z", "bbox_min_x", "bbox_min_y", "bbox_min_z", "bbox_max_x", "bbox_max_y", "bbox_max_z",
             "obb_center_x", "obb_center_y", "obb_center_z", "obb_min_x", "obb_min_y", "obb_min_z", "obb_max_x", "obb_max_y", "obb_max_z"]
nodes_df = nodes_df.drop(columns=[c for c in drop_cols if c in nodes_df.columns])

geom_cols = ["centroid_x", 
             "centroid_y", 
             "centroid_z",
             "obb_extent_x", 
             "obb_extent_y", 
             "obb_extent_z",
             "angle_to_x", 
             "angle_to_y"]

has_geom = nodes_df["obb_extent_x"].notna().sum()

print(f"Nodes extracted: {len(nodes_df)}")
print(f"With geometry:   {has_geom} / {len(nodes_df)}  ({100*has_geom/len(nodes_df):.0f}%)")
print(f"\nGeometry columns kept: {geom_cols}")
print(f"\nIFC types in nodes_df:")
print(nodes_df["ifc_type"].value_counts().to_string())

enrichment_cols = ["wall_long_axis", "host_wall_id"]
for col in enrichment_cols:
    if col in nodes_df.columns:
        filled = nodes_df[col].notna().sum()
        print(f"\n{col}: {filled} elements enriched")

print(f"\nAll columns: {list(nodes_df.columns)}")
nodes_df.head(10)

Extracting mesh geometry:   0%|          | 0/289 [00:00<?, ?it/s]

Extracting mesh geometry: 100%|██████████| 289/289 [00:06<00:00, 47.42it/s] 


Total relations: 325
Nodes extracted: 289
With geometry:   286 / 289  (99%)

Geometry columns kept: ['centroid_x', 'centroid_y', 'centroid_z', 'obb_extent_x', 'obb_extent_y', 'obb_extent_z', 'angle_to_x', 'angle_to_y']

IFC types in nodes_df:
ifc_type
IfcFurnishingElement    61
IfcWallStandardCase     56
IfcOpeningElement       50
IfcWindow               24
IfcSlab                 21
IfcSpace                21
IfcDoor                 14
IfcCovering             13
IfcBeam                  8
IfcFooting               7
IfcMember                4
IfcRailing               4
IfcStairFlight           2
IfcStair                 2
IfcRoof                  1
IfcWall                  1

wall_long_axis: 57 elements enriched

host_wall_id: 36 elements enriched

All columns: ['ifc_id', 'name', 'ifc_type', 'centroid_x', 'centroid_y', 'centroid_z', 'obb_extent_x', 'obb_extent_y', 'obb_extent_z', 'angle_to_x', 'angle_to_y', 'wall_long_axis', 'host_wall_id']


,ifc_id,name,ifc_type,centroid_x,centroid_y,centroid_z,obb_extent_x,obb_extent_y,obb_extent_z,angle_to_x,angle_to_y,wall_long_axis,host_wall_id
0,36686,M_W-Wide Flange:W310X60:W310X60:207325,IfcBeam,4.400000,-13.710650,2.9485,3.710650,0.151500,0.1015,90.000000,180.000000,<NA>,<NA>
1,36892,M_W-Wide Flange:W410X60:W410X60:208949,IfcBeam,3.357556,-17.523000,2.8965,3.090944,0.203500,0.0890,0.000000,90.000000,<NA>,<NA>
2,37065,M_W-Wide Flange:W410X60:W410X60:209121,IfcBeam,0.266613,-15.462360,5.7965,2.060640,0.203500,0.0890,90.000000,0.000000,<NA>,<NA>
3,37195,M_W-Wide Flange:W410X60:W410X60:209166,IfcBeam,6.478954,-17.528767,5.7965,2.050000,0.203500,0.0890,0.000000,90.000000,<NA>,<NA>
4,37325,M_W-Wide Flange:W310X60:W310X60:209260,IfcBeam,4.400000,-4.089350,2.9485,3.710650,0.151500,0.1015,90.000000,0.000000,<NA>,<NA>
5,37456,M_W-Wide Flange:W410X60:W410X60:209261,IfcBeam,5.442444,-0.277000,2.8965,3.090944,0.203500,0.0890,0.000000,90.000000,<NA>,<NA>
6,37586,M_W-Wide Flange:W410X60:W410X60:209262,IfcBeam,8.533387,-2.337640,5.7965,2.060640,0.203500,0.0890,90.000000,180.000000,<NA>,<NA>
7,37716,M_W-Wide Flange:W410X60:W410X60:209263,IfcBeam,2.321046,-0.271233,5.7965,2.050000,0.203500,0.0890,0.000000,90.000000,<NA>,<NA>
8,23671,Compound Ceiling:Gypsum Board:187149,IfcCovering,2.904486,-5.034000,2.6285,5.133629,4.482668,0.0285,98.856155,171.143845,<NA>,<NA>
9,23768,Compound Ceiling:Gypsum Board:187462,IfcCovering,3.349500,-8.701000,2.6285,1.024000,0.775500,0.0285,90.000000,180.000000,<NA>,<NA>


### 3.3 Semantic Data: Property Sets & Quantities

Besides geometry, IFC elements also have **Property Sets (Psets)** and **Quantity Take-offs (QTOs)**. These are key–value pairs that describe things like material, fire rating, thermal properties, and dimensions.

In this notebook we use them to:

1. Enrich node features (for example: `IsExternal`, `FireRating`, `ThermalTransmittance`).
2. Provide labels for supervised learning tasks (for example: predicting a fire rating class).
3. Export properties into RDF so they can be checked by SHACL constraints (Section 8).

**Sparse vs. dense features.**  
Not every element has the same Psets and parameters, so the resulting feature tables are sparse. There are several ways to handle this:

- fill missing values (imputation),
- keep only a common subset of well-filled properties,
- or give each node type its own feature space (which fits naturally with `HeteroData`).


In [5]:
params_df = extractor.extract_parameters_df(ifc_file, include_qto=True)

print(f"Parameter records:  {len(params_df)}")
print(f"Unique Psets:       {params_df['pset'].nunique()}")
print(f"Unique parameters:  {params_df['param'].nunique()}")
print(f"Element types with params: {params_df['ifc_type'].nunique()}\n")

print("Top 15 Property Sets by frequency:")
print(params_df["pset"].value_counts().head(15).to_string())

print("\n\nSample records:")
params_df.sample(10, random_state=42)

Extracting parameters: 100%|██████████| 289/289 [00:00<00:00, 531.18it/s]

Parameter records:  14236
Unique Psets:       34
Unique parameters:  271
Element types with params: 16

Top 15 Property Sets by frequency:
pset
PSet_Revit_Type_Other                     6143
PSet_Revit_Other                          1675
PSet_Revit_Constraints                    1327
PSet_Revit_Type_Identity Data              955
_Intrinsic                                 834
PSet_Revit_Type_Dimensions                 543
PSet_Revit_Type_Construction               465
PSet_Revit_Dimensions                      464
PSet_Revit_Type_Materials and Finishes     285
PSet_Revit_Phasing                         246
Pset_WallCommon                            228
PSet_Revit_Structural Analysis             147
PSet_Revit_Structural                      143
PSet_Revit_Type_Graphics                   108
PSet_Revit_Identity Data                   101


Sample records:


,ifc_id,ifc_type,pset,param,value,value_type
8918,22084,IfcWindow,PSet_Revit_Type_Construction,Wall Closure,0,int
8452,7743,IfcWindow,PSet_Revit_Other,TagNumber,TagNumber,str
1078,24128,IfcCovering,PSet_Revit_Type_Other,ExpectedLife,ExpectedLife,str
2483,23408,IfcFooting,PSet_Revit_Type_Other,Shape,Shape,str
4463,5399,IfcWall,PSet_Revit_Type_Other,WarrantyGuarantorLabor,WarrantyGuarantorLabor,str
8242,7025,IfcWindow,PSet_Revit_Type_Other,ModelLabel,ModelLabel,str
13453,36526,IfcFurnishingElement,PSet_Revit_Type_Identity Data,Manufacturer,Manufacturer,str
9279,22344,IfcWindow,PSet_Revit_Type_Other,ExpectedLife,ExpectedLife,str
4981,4818,IfcWallStandardCase,PSet_Revit_Type_Other,CodePerformance,CodePerformance,str
8296,7190,IfcWindow,PSet_Revit_Type_Other,Shape,Shape,str


In [6]:
wall_params = params_df[params_df["ifc_type"] == "IfcWallStandardCase"].copy()
if wall_params.empty:
    wall_params = params_df[params_df["ifc_type"].str.contains("Wall", case=False)].copy()

if not wall_params.empty:
    wall_params["col"] = wall_params["pset"] + "." + wall_params["param"]
    wall_wide = wall_params.pivot_table(
        index="ifc_id", columns="col", values="value", aggfunc="first"
    )
    print(f"Wall wide table: {wall_wide.shape[0]} walls × {wall_wide.shape[1]} properties")
    non_null_pct = wall_wide.notna().mean().sort_values(ascending=False)
    print(f"\nTop 20 properties by fill rate:")
    print((non_null_pct.head(20) * 100).round(1).to_string())
    print(f"\n(Showing first 5 rows of the wide table)")
    display(wall_wide.iloc[:5, :10])
else:
    print("No wall elements found in params_df")

Wall wide table: 56 walls × 71 properties

Top 20 properties by fill rate:
col
PSet_Revit_Constraints.Base Constraint            100.0
PSet_Revit_Constraints.Base Extension Distance    100.0
PSet_Revit_Constraints.Base Offset                100.0
PSet_Revit_Constraints.Location Line              100.0
PSet_Revit_Constraints.Base is Attached           100.0
PSet_Revit_Constraints.Related to Mass            100.0
PSet_Revit_Constraints.Room Bounding              100.0
PSet_Revit_Type_Other.Classification Code         100.0
PSet_Revit_Constraints.Top Extension Distance     100.0
PSet_Revit_Constraints.Top Offset                 100.0
PSet_Revit_Constraints.Top is Attached            100.0
PSet_Revit_Dimensions.Length                      100.0
PSet_Revit_Constraints.Unconnected Height         100.0
PSet_Revit_Other.AssetIdentifier                  100.0
PSet_Revit_Dimensions.Volume                      100.0
PSet_Revit_Other.SerialNumber                     100.0
PSet_Revit_Other.TagNumbe

col,PSet_Revit_Analytical Model.Enable Analytical Model,PSet_Revit_Constraints.Base Constraint,PSet_Revit_Constraints.Base Extension Distance,PSet_Revit_Constraints.Base Offset,PSet_Revit_Constraints.Base is Attached,PSet_Revit_Constraints.Location Line,PSet_Revit_Constraints.Related to Mass,PSet_Revit_Constraints.Room Bounding,PSet_Revit_Constraints.Top Constraint,PSet_Revit_Constraints.Top Extension Distance
ifc_id,,,,,,,,,,
3797,NaN,Level 1,0.0,0.0,False,2,False,True,Up to level: Level 2,0.0
3999,NaN,Level 1,0.0,0.0,False,2,False,True,Up to level: Level 2,0.0
4043,NaN,Level 1,0.0,0.0,False,2,False,True,Up to level: Level 2,0.0
4087,NaN,Level 1,0.0,0.0,False,2,False,True,Up to level: Level 2,0.0
4131,NaN,Level 1,0.0,0.0,False,2,False,True,Up to level: Level 2,0.0


### 3.4 Relation Extraction: IFC Edges

IFC defines explicit **relationships** between elements as separate objects. In this notebook we extract three core relation types:

| IFC Relation              | Meaning                          | Example          |
|---------------------------|----------------------------------|------------------|
| `IfcRelSpaceBoundary`     | A space is bounded by an element | Room ↔ Wall      |
| `IfcRelVoidsElement`      | An opening cuts through an element | Wall ↔ Opening |
| `IfcRelFillsElement`      | A building element fills an opening | Opening ↔ Door/Window |

In the graph, these relationships become **typed edges**. This keeps the IFC semantics intact instead of collapsing everything into a simple, untyped adjacency matrix.

**Why use IFC relations and not just distances?**  
Simple geometric proximity (e.g. k‑NN in 3D) says that two elements are near each other, but not *why*. IFC relations encode **design intent**: a door is linked to a wall because the architect placed it in that wall, not just because it is nearby. This intent is lost if we rely only on distance-based edges.

**Scope and possible extensions.**  
Here we focus on three relations that form a clear subgraph connecting spaces, walls, openings and infill elements, which is enough for the tasks in this notebook. The IFC schema also has other relation classes that could be added later:

| IFC Relation                  | What it adds                               | Why it is useful                           |
|------------------------------|--------------------------------------------|--------------------------------------------|
| `IfcRelContainedInSpatialStructure` | Element–storey/building hierarchy      | Multi-scale reasoning, floor-level analysis|
| `IfcRelAggregates`           | Part–whole decomposition (e.g. stair → flights) | Composite element modelling           |
| `IfcRelConnectsElements`     | Physical element–element connections       | Structural continuity, load-path reasoning |
| `IfcRelAssociatesMaterial`   | Element–material links                     | Material-aware predictions (thermal, fire) |

Adding these would increase the number of edge types in the graph and enable tasks that need hierarchical or material context. In this notebook we keep them as a clear extension point.


In [7]:
edges_df = edges_df_raw

print(f"Total edges: {len(edges_df)}\n")
print("Edges by relation type:")
print(edges_df["relation"].value_counts().to_string())

if "boundary_type" in edges_df.columns:
    bt_counts = edges_df["boundary_type"].value_counts()
    if not bt_counts.empty:
        print(f"\nBoundary types (IfcRelSpaceBoundary only):")
        print(bt_counts.to_string())

node_ids_in_edges = set(edges_df["source"]).union(set(edges_df["target"]))
node_ids_in_nodes = set(nodes_df["ifc_id"])
orphan_ids = node_ids_in_edges - node_ids_in_nodes
print(f"\nEdge endpoints not in nodes_df: {len(orphan_ids)}")

print("\nSample edges:")
edges_df.head(10)

Total edges: 325

Edges by relation type:
relation
IfcRelSpaceBoundary    237
IfcRelVoidsElement      50
IfcRelFillsElement      38

Boundary types (IfcRelSpaceBoundary only):
boundary_type
INTERNAL    175
EXTERNAL     62

Edge endpoints not in nodes_df: 0

Sample edges:


,source,target,relation,boundary_type
0,514,5267,IfcRelSpaceBoundary,INTERNAL
1,514,6652,IfcRelSpaceBoundary,EXTERNAL
2,514,21401,IfcRelSpaceBoundary,INTERNAL
3,514,4508,IfcRelSpaceBoundary,INTERNAL
4,514,4508,IfcRelSpaceBoundary,INTERNAL
5,514,5267,IfcRelSpaceBoundary,INTERNAL
6,514,4043,IfcRelSpaceBoundary,EXTERNAL
7,514,5267,IfcRelSpaceBoundary,INTERNAL
8,514,3999,IfcRelSpaceBoundary,EXTERNAL
9,514,4287,IfcRelSpaceBoundary,EXTERNAL


### 3.5 Quick Sanity Check: Build & Visualise the Raw Graph

In [8]:
G_raw = build_ifc_graph(nodes_df, edges_df)

print(f"NetworkX graph: {G_raw.number_of_nodes()} nodes, {G_raw.number_of_edges()} edges\n")

report = validate_ifc_graph(nodes_df, edges_df, G_raw)

print("=" * 60)
print("IFC GRAPH VALIDATION REPORT")
print("=" * 60)
for k, v in report.items():
    print(f"  {k:>35s}: {v}")
print("=" * 60)

edges kept by relation: {'IfcRelSpaceBoundary': 237, 'IfcRelFillsElement': 38, 'IfcRelVoidsElement': 48}
NetworkX graph: 286 nodes, 278 edges

IFC GRAPH VALIDATION REPORT
                              n_nodes: 286
                              n_edges: 278
                             isolates: 96
                           self_loops: 0
                  parallel_duplicates: 0
              edges_ref_missing_nodes: 0
                dup_guids_in_nodes_df: 0
                         n_components: 111
                    largest_component: 158
          centroid_x_outliers_abs_z>6: 0
               nodes_missing_ifc_type: 0
               edges_missing_relation: 0


In [9]:
fig = plot_ifc_graph_3d(G_raw, title="Duplex.ifc — Raw IFC Graph (3D)", size=5)
fig.show()

Clean Ifc graph -  keep only largest connected part

In [10]:
G_before_clean = build_ifc_graph(nodes_df, edges_df)

components, lcc_stats = separate_disconnected_components(
    G_before_clean,
    large_threshold=1,
    return_large_only=False,
)

G_raw = max(components, key=lambda g: g.number_of_nodes()) if components else G_before_clean

print(
    f"After LCC cleaning: {G_raw.number_of_nodes()} nodes, {G_raw.number_of_edges()} edges "
    f"(from {lcc_stats['total_components']} components; largest={lcc_stats['largest_component_size']})"
)

edges kept by relation: {'IfcRelSpaceBoundary': 237, 'IfcRelFillsElement': 38, 'IfcRelVoidsElement': 48}
After LCC cleaning: 158 nodes, 260 edges (from 111 components; largest=158)


In [11]:
clean_ids = set(G_raw.nodes())

data_cleaned = {
    "nodes_df": nodes_df[nodes_df["ifc_id"].isin(clean_ids)].copy(),
    "edges_df": edges_df[
        edges_df["source"].isin(clean_ids) & edges_df["target"].isin(clean_ids)
    ].copy(),
}

if "params_df" in globals():
    data_cleaned["params_df"] = params_df[params_df["ifc_id"].isin(clean_ids)].copy()

nodes_df = data_cleaned["nodes_df"]
edges_df = data_cleaned["edges_df"]
if "params_df" in data_cleaned:
    params_df = data_cleaned["params_df"]

print(
    f"Cleaned tables ready: nodes_df={len(nodes_df)}, edges_df={len(edges_df)}"
    + (f", params_df={len(params_df)}" if "params_df" in data_cleaned else "")
)

Cleaned tables ready: nodes_df=158, edges_df=305, params_df=7100


### 3.6 Section Summary

At this point we have three core DataFrames extracted from the IFC file:

| DataFrame   | Rows                           | Key columns                                                           | Purpose                                         |
|-------------|--------------------------------|------------------------------------------------------------------------|-------------------------------------------------|
| `nodes_df`  | one per IFC element / space    | `ifc_id`, `ifc_type`, `centroid_*`, `obb_extent_*`, `angle_to_*`, `wall_long_axis`, `host_wall_id` | Graph nodes with rotation-aware geometric features (no AABB) |
| `edges_df`  | one per IFC relationship       | `source`, `target`, `relation`                                        | Graph edges that preserve IFC design intent     |
| `params_df` | one per (element, property)    | `ifc_id`, `pset`, `param`, `value`                                    | Semantic enrichment and labels                  |

The extraction pipeline reuses `IFCMeshGeometryDataExtractor` from my master’s thesis codebase. In addition to basic AABB geometry, it computes **PCA-based Oriented Bounding Boxes** (`_compute_obb`), which provide rotation-invariant dimensions and orientation angles. After extraction, `_add_wall_long_axis` and `_add_host_wall_id` further enrich `nodes_df` with wall direction and hosting relationships.

**From the validation report:** the numbers of isolated nodes and connected components are important signals. They guide graph cleaning and pruning. In the further pipeline we keep only the **largest connected component** of the graph, because:

- GNNs learn patterns by passing messages along edges; tiny components and isolated nodes provide almost no context.
- Small components are often artefacts of partial modelling or import errors rather than meaningful design subgraphs.
- Focusing on the largest component gives a coherent building-scale context and a more stable training signal for the GNN.


---

## 4. Geometric Representations: Multi-Level Encoding

So far, each graph node has only OBB-based geometry (centroid, extents, angles). This is compact, but it does not fully describe the **shape** of the element. For example, a rectangular wall and an L‑shaped wall can have similar OBB extents but very different surfaces.

To capture more shape information, we build an extra **shape embedding** from the element’s point cloud and add it as an additional node feature.

### 4.1 Representation Levels

We end up with several complementary levels of representation:

| Level             | What it captures                    | Encoding                                     | Dim |
|-------------------|-------------------------------------|----------------------------------------------|-----|
| **Graph edges**   | How elements are related            | IFC relations → typed edges                  | –   |
| **OBB features**  | Position, size, orientation         | Centroid + OBB extents + angles              | 8   |
| **Shape embedding** | Surface geometry of the element   | Point cloud → shape descriptors → PCA        | 7   |
| **Name embedding**  | Semantic hints from element names | (Optional) Name text → text encoder → PCA    | 7   |

In the GNN, these feature blocks are concatenated into one node feature vector.

### 4.2 Point Cloud → Shape Embedding

For each IFC element we:

1. **Sample a point cloud**  
   Take a fixed number of points (N = 256) from the mesh vertices and shift them so the cloud is centred at the origin.

2. **Compute simple shape descriptors**  
   From this cloud we compute 13 statistics such as:
   - PCA eigenvalues and their ratios (elongation / flatness / compactness),
   - distance (radius) statistics: mean, standard deviation, max, quartiles,
   - vertical spread.

3. **Reduce to 7 dimensions with PCA**  
   We standardise these 13 descriptors and apply PCA to obtain a 7‑dimensional shape embedding.

**Why descriptors instead of just flattening all points?**  
Flattening 256×3 coordinates into a 768‑dimensional vector is too high‑dimensional and depends on point order. In contrast, shape descriptors are order‑invariant, geometrically meaningful, and compact. After that, 7 PCA components already capture most of the useful variation.

**Why not use PointNet?**  
A learned encoder like PointNet would be more powerful, but it needs labelled data and a separate training loop. Here we use a lightweight baseline (descriptors + PCA) that works out‑of‑the‑box and keeps the focus on data representation. A PointNet‑style encoder would be a natural future extension.


### 4.3 Sampling & Encoding

In [12]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

SHAPE_EMB_DIM = 7

def _pc_descriptors(pc):
    dists = np.linalg.norm(pc, axis=1)
    eigvals = np.sort(np.linalg.eigvalsh(np.cov(pc.T)))[::-1]
    ev = np.maximum(eigvals, 1e-12)
    return np.array([
        *eigvals, ev[0]/ev[1], ev[1]/ev[2], ev[2]/ev[0],
        dists.mean(), dists.std(), dists.max(),
        np.percentile(dists, 25), np.percentile(dists, 75),
        pc[:, 2].std(), np.abs(pc[:, 2]).mean(),
    ], dtype=np.float32)

ids = list(pc_raw.keys())
print(f"pc_raw has {len(ids)} entries")
assert len(ids) > 0, "pc_raw is empty — re-run cell 7 (geometry extraction) first!"
X = StandardScaler().fit_transform(np.stack([_pc_descriptors(pc_raw[i]) for i in ids]))
X_emb = PCA(n_components=SHAPE_EMB_DIM, random_state=42).fit_transform(X)

pca_var = PCA(n_components=SHAPE_EMB_DIM, random_state=42).fit(X).explained_variance_ratio_.sum()
print(f"PCA {X.shape[1]} -> {SHAPE_EMB_DIM} dim | explained variance: {pca_var:.1%}")

emb_cols = [f"shape_emb_{i}" for i in range(SHAPE_EMB_DIM)]
emb_df = pd.DataFrame(X_emb, columns=emb_cols)
emb_df["ifc_id"] = ids

nodes_df = nodes_df.drop(columns=[c for c in nodes_df.columns if c.startswith("shape_emb")], errors="ignore")
nodes_df = nodes_df.merge(emb_df, on="ifc_id", how="left")

nodes_df[["ifc_id", "ifc_type", "name"] + emb_cols].head(10)

pc_raw has 286 entries
PCA 13 -> 7 dim | explained variance: 96.3%


,ifc_id,ifc_type,name,shape_emb_0,shape_emb_1,shape_emb_2,shape_emb_3,shape_emb_4,shape_emb_5,shape_emb_6
0,23671,IfcCovering,Compound Ceiling:Gypsum Board:187149,3.346585,-2.367169,1.695219,-1.628582,0.339490,-0.718265,-1.124994
1,23768,IfcCovering,Compound Ceiling:Gypsum Board:187462,-0.989223,-1.396158,-0.447685,0.163227,-0.174869,0.351792,-0.913399
2,23826,IfcCovering,Compound Ceiling:Gypsum Board:187470,-0.989159,-1.396140,-0.447949,0.163381,-0.174919,0.351612,-0.913173
3,23884,IfcCovering,Compound Ceiling:Gypsum Board:187483,-0.941357,-1.399508,-0.463805,0.177790,-0.177248,0.348680,-0.904265
4,23992,IfcCovering,Compound Ceiling:Gypsum Board:187508,3.745076,-2.445920,2.036845,-1.788094,0.418488,-0.312074,-1.381961
5,24060,IfcCovering,Compound Ceiling:Gypsum Board:187639,1.911001,-1.947639,0.363657,-0.527055,0.042099,-0.321284,-0.856621
6,24128,IfcCovering,Compound Ceiling:Gypsum Board:187649,1.812951,-1.926266,0.310716,-0.506656,0.028193,-0.384262,-0.825130
7,24186,IfcCovering,Compound Ceiling:Gypsum Board:187659,-0.021544,-1.478810,-0.705887,0.422965,-0.205532,0.353359,-0.758426
8,24268,IfcCovering,Compound Ceiling:Gypsum Board:187667,0.785934,-1.805097,0.180406,-0.585651,-0.023814,-0.963860,-0.658738
9,24326,IfcCovering,Compound Ceiling:Gypsum Board:187675,-0.021339,-1.478846,-0.706395,0.423451,-0.205622,0.352908,-0.758124


### 4.4 Visualisation: Element Point Clouds

A few sampled point clouds to illustrate the shape information that gets compressed into 7-dim embeddings.

In [13]:
show_types = ["IfcWallStandardCase", "IfcDoor", "IfcSpace"]
samples = {
    t: int(nodes_df.loc[(nodes_df["ifc_type"] == t) & nodes_df["ifc_id"].isin(pc_raw), "ifc_id"].iloc[0])
    for t in show_types
    if not nodes_df.loc[(nodes_df["ifc_type"] == t) & nodes_df["ifc_id"].isin(pc_raw), "ifc_id"].empty
}

fig = make_subplots(
    rows=1, cols=len(samples),
    specs=[[{"type": "scatter3d"}] * len(samples)],
    subplot_titles=[f"{t} (id={s}) — {pc_raw[s].shape[0]} pts" for t, s in samples.items()],
)
for i, (t, sid) in enumerate(samples.items(), 1):
    pc = pc_raw[sid]
    fig.add_trace(
        go.Scatter3d(
            x=pc[:, 0], y=pc[:, 1], z=pc[:, 2],
            mode="markers",
            marker=dict(size=3, opacity=0.85),
            name=t,
        ),
        row=1, col=i,
    )
for k in range(1, len(samples) + 1):
    scene_key = "scene" if k == 1 else f"scene{k}"
    fig.update_layout(**{scene_key: dict(aspectmode="data")})

fig.update_layout(
    title="Sampled Point Clouds (256 pts, centred, real proportions)",
    height=500, width=450 * len(samples), showlegend=False,
)
fig.show()

---

## 5. Graph Construction: Heterogeneous Property Graph

### 5.1 Why Use `HeteroData` Instead of a Homogeneous Graph?

A standard PyG `Data` object assumes that all nodes and all edges are of the same type. In practice this means:

- all nodes share one big feature matrix,
- edge type has to be encoded as just another feature,
- the GNN uses the same weights for all relations.

For BIM this is not ideal, because walls, doors and spaces play very different roles.

`HeteroData` lets us keep type information directly in the graph:

| Aspect          | Homogeneous `Data`                     | Heterogeneous `HeteroData`                            |
|-----------------|----------------------------------------|-------------------------------------------------------|
| Node features   | One matrix for all nodes               | Separate `x` per node type                            |
| Edge indices    | One global `edge_index`                | Separate index per `(src_type, rel_type, dst_type)`   |
| Message passing | Shared weights for all edges           | Relation- and type-aware weights                      |
| IFC semantics   | Partly flattened into features         | Reflected in the graph schema itself                  |

For this notebook, `HeteroData` is mainly used to clearly show the IFC structure. In the master’s pipeline I also successfully use a homogeneous R-GCN with relation types.

### 5.2 Node Types: Use Raw IFC Classes

I keep IFC classes as they are:

- node type = `ifc_type` from the IFC file (e.g. `IfcWallStandardCase`, `IfcDoor`, `IfcWindow`, `IfcSpace`).

This avoids custom naming and makes it easy to map results back to the original BIM model.

### 5.3 Edge Types: Use Raw IFC Relation Names

I also keep the relation names:

- edge type = IFC relation name (e.g. `IfcRelSpaceBoundary`, `IfcRelVoidsElement`, `IfcRelFillsElement`).

This keeps the graph as close as possible to the original IFC schema and minimises extra conversion logic before modelling.


In [14]:
import torch
from torch_geometric.data import HeteroData

FEAT_COLS = [
    "centroid_x", "centroid_y", "centroid_z",
    "obb_extent_x", "obb_extent_y", "obb_extent_z",
    "angle_to_x", "angle_to_y",
] + [f"shape_emb_{i}" for i in range(7)]

nodes_df["node_type"] = nodes_df["ifc_type"].fillna("UnknownIfcType")

hetero = HeteroData()
global_to_local = {}

for ntype, grp in nodes_df.groupby("node_type", sort=True):
    ids = grp["ifc_id"].astype(int).tolist()
    hetero[ntype].x = torch.tensor(grp[FEAT_COLS].fillna(0).to_numpy(np.float32))
    hetero[ntype].ifc_id = ids
    hetero[ntype].name = grp["name"].fillna("").tolist()
    global_to_local.update({ifc_id: (ntype, i) for i, ifc_id in enumerate(ids)})

edge_rows = edges_df[["source", "target", "relation"]].dropna().drop_duplicates()
duplicates_removed = len(edges_df) - len(edge_rows)

edge_triples = {}
_skipped = 0
for src, tgt, rel in edge_rows.itertuples(index=False):
    src_loc = global_to_local.get(int(src))
    tgt_loc = global_to_local.get(int(tgt))
    if src_loc is None or tgt_loc is None:
        _skipped += 1
        continue

    src_type, src_idx = src_loc
    tgt_type, tgt_idx = tgt_loc
    triple = (src_type, rel, tgt_type)
    edge_triples.setdefault(triple, [[], []])
    edge_triples[triple][0].append(src_idx)
    edge_triples[triple][1].append(tgt_idx)

for triple, (src_idx, tgt_idx) in edge_triples.items():
    hetero[triple].edge_index = torch.tensor([src_idx, tgt_idx], dtype=torch.long)

print("=== HeteroData Summary ===\n")
print("Node types:")
for ntype in sorted(hetero.node_types):
    print(f"  {ntype:22s} {hetero[ntype].x.shape[0]:4d} nodes  x  {hetero[ntype].x.shape[1]} features")

print(f"\nEdge types ({_skipped} skipped, {duplicates_removed} duplicates removed):")
for etype in hetero.edge_types:
    print(f"  {str(etype):75s} {hetero[etype].edge_index.shape[1]:4d} edges")

print(f"\n{hetero}")

=== HeteroData Summary ===

Node types:
  IfcCovering              13 nodes  x  15 features
  IfcDoor                  14 nodes  x  15 features
  IfcOpeningElement        36 nodes  x  15 features
  IfcSlab                  18 nodes  x  15 features
  IfcSpace                 20 nodes  x  15 features
  IfcWall                   1 nodes  x  15 features
  IfcWallStandardCase      34 nodes  x  15 features
  IfcWindow                22 nodes  x  15 features

Edge types (0 skipped, 45 duplicates removed):
  ('IfcSpace', 'IfcRelSpaceBoundary', 'IfcSlab')                                40 edges
  ('IfcSpace', 'IfcRelSpaceBoundary', 'IfcDoor')                                25 edges
  ('IfcSpace', 'IfcRelSpaceBoundary', 'IfcWallStandardCase')                    80 edges
  ('IfcSpace', 'IfcRelSpaceBoundary', 'IfcCovering')                            23 edges
  ('IfcSpace', 'IfcRelSpaceBoundary', 'IfcWindow')                              16 edges
  ('IfcSpace', 'IfcRelSpaceBoundary', 'IfcWall')   

### 5.4 Graph Visualisation

In [15]:
G_vis = build_ifc_graph(nodes_df, edge_rows)
fig = plot_ifc_graph_3d(G_vis, title="HeteroData graph (raw IFC types/relations)", size=5)
fig.show()

edges kept by relation: {'IfcRelSpaceBoundary': 188, 'IfcRelFillsElement': 36, 'IfcRelVoidsElement': 36}


### 5.5 Graph Statistics & Sanity Checks

Before proceeding to modelling, we validate the graph structure:
- No self-loops
- No isolated nodes (every element participates in at least one relation)
- Connected component analysis
- Degree distribution per node type

In [16]:
_ = validate_ifc_graph(nodes_df, edge_rows, G_vis)

rel_counts = (
    edge_rows["relation"]
    .value_counts()
    .rename_axis("relation")
    .reset_index(name="count")
    .sort_values("count", ascending=True)
)

fig_bar = px.bar(
    rel_counts,
    x="count",
    y="relation",
    orientation="h",
    title="Edge count by IFC relation",
)
fig_bar.update_layout(width=700, height=380)
fig_bar.show()

---

## 6. Feature Engineering (Summary)

By the time we build `HeteroData` (Section 5), each node already has a **15‑dimensional** feature vector:

| Feature group          | Dim | Source                         |
|------------------------|:---:|--------------------------------|
| Spatial position       |  3  | Centroid from OBB geometry     |
| OBB extents            |  3  | Oriented bounding box (PCA)    |
| Orientation angles     |  2  | Rotation relative to global axes |
| Shape embedding        |  7  | PCA over point‑cloud descriptors |

So each element is described by where it is, how big it is, how it is oriented, and a compact summary of its shape.

**What could be added in a larger production setup** (not implemented end‑to‑end in this demo, but natural extensions of the same idea):

- **Semantic Psets** — properties like `FireRating`, `IsExternal`, `LoadBearing` from IFC Property Sets  
  (we extract them in Section 3.3, but in Duplex.ifc they are mostly sparse).
- **Name embeddings (optional)** — element names and type labels often contain useful cues  
  (e.g. *“Fire Door EI30”* vs *“Interior Door”*). In a multi‑project setting these strings can be encoded with a text model (e.g. SentenceTransformer) and reduced with PCA, then added as extra node features.
- **Edge features** — for example, `Internal` / `External` from `IfcRelSpaceBoundary` as a one‑hot edge attribute.
- **Derived geometric features** — such as `volume = extent_x * extent_y * extent_z`,  
  `aspect_ratio = max_extent / min_extent`, etc.

**Normalisation.**  
For real multi‑building training, geometric features would typically be z‑score normalised per graph; boolean / categorical features can stay as booleans or be label‑encoded. For this single‑building demonstration, working with raw (unnormalised) values is sufficient to illustrate the representation.


In [17]:
print("Feature matrices already assigned in Section 5:\n")
for nt in sorted(hetero.node_types):
    print(f"  hetero['{nt}'].x  →  {tuple(hetero[nt].x.shape)}")
print(f"\nFeature columns: {FEAT_COLS}")

Feature matrices already assigned in Section 5:

  hetero['IfcCovering'].x  →  (13, 15)
  hetero['IfcDoor'].x  →  (14, 15)
  hetero['IfcOpeningElement'].x  →  (36, 15)
  hetero['IfcSlab'].x  →  (18, 15)
  hetero['IfcSpace'].x  →  (20, 15)
  hetero['IfcWall'].x  →  (1, 15)
  hetero['IfcWallStandardCase'].x  →  (34, 15)
  hetero['IfcWindow'].x  →  (22, 15)

Feature columns: ['centroid_x', 'centroid_y', 'centroid_z', 'obb_extent_x', 'obb_extent_y', 'obb_extent_z', 'angle_to_x', 'angle_to_y', 'shape_emb_0', 'shape_emb_1', 'shape_emb_2', 'shape_emb_3', 'shape_emb_4', 'shape_emb_5', 'shape_emb_6']


---

## 7. GNN Model

- preparing the graph representation,
- defining node features and edge types,
- and outlining the model architecture that would be used once a multi‑building dataset is available.

---

### 7.1 Homogeneous vs Heterogeneous Graphs (Short Summary)

We build a `HeteroData` graph mainly to show how IFC semantics can be represented explicitly. For a production BIM pipeline, a **homogeneous R‑GCN with relation types and `ifc_type` encoded in the node features** is often a practical choice, as it:

- directly supports multiple relation types,
- keeps all nodes in a shared feature space,
- uses a compact number of parameters.

`HeteroData` remains useful for analysis, visualisation, and cases where different node types require different feature spaces.

---

### 7.2 Model Architecture Used in the Master’s Pipeline

The master’s thesis uses the following R‑GCN for node classification:

```text
Input: x ∈ ℝ^(N × F), edge_index, edge_type
    ↓
RGCNConv(F → 64, num_relations = R) + ReLU + Dropout(0.3)
    ↓
RGCNConv(64 → 32, num_relations = R) + ReLU
    ↓
Linear(32 → C) → softmax
```

- **Task:** node classification (e.g. predict door fire‑rating class).
- **Loss:** cross‑entropy with class weights for imbalance.
- **Optimiser:** Adam (lr = 0.01, weight_decay = 5e‑4).

This architecture can be applied directly once a multi‑building dataset is available, using the graph representation prepared in this notebook.
```

In [18]:
from torch_geometric.nn import RGCNConv
import torch.nn as nn
import torch.nn.functional as F

class BIM_RGCN(nn.Module):
    """Minimal 2-layer R-GCN for node classification on a BIM graph."""
    def __init__(self, in_dim, hidden_dim, out_dim, num_relations):
        super().__init__()
        self.conv1 = RGCNConv(in_dim, hidden_dim, num_relations)
        self.conv2 = RGCNConv(hidden_dim, out_dim, num_relations)

    def forward(self, x, edge_index, edge_type):
        x = F.relu(self.conv1(x, edge_index, edge_type))
        x = F.dropout(x, p=0.3, training=self.training)
        return self.conv2(x, edge_index, edge_type)

print(f"BIM_RGCN defined — not trained (single-building demo).")
print(f"In production: trained via LOBO CV on 9+ buildings, F1 up to 0.996.")

BIM_RGCN defined — not trained (single-building demo).
In production: trained via LOBO CV on 9+ buildings, F1 up to 0.996.


### 7.4 Training & Evaluation

In this notebook we do **not** train a GNN. One small building (289 nodes) is not enough to obtain meaningful learning curves or generalisable metrics. Here the focus is on the **data representation and pipeline design**, not on model performance.

In my master’s thesis pipeline, training and evaluation are done on 9+ buildings using a **Leave-One-Building-Out (LOBO)** protocol:

- **Split:** hold out one building as test, train on the remaining buildings.
- **Loss:** weighted cross‑entropy to handle class imbalance (for example, many doors without fire rating).
- **Early stopping:** based on validation loss, with patience of 20 epochs.
- **Results:** macro‑F1 in the range ~0.80–0.95 depending on the specific task, with confusion matrices showing strong diagonal dominance.

### 7.5 Key Takeaway

**Summary:** Heterogeneous graphs make node and edge types explicit, which is useful for explaining the representation. However, recent BIM‑specific work (e.g. BIGNet) shows that **homogeneous graphs with relation‑aware convolutions (R‑GCN)** often perform better for BIM feature learning, because all node types share one feature space and can transfer knowledge between each other. In this notebook we build `HeteroData` for clarity and teaching purposes, while the production‑oriented pipeline uses a homogeneous R‑GCN setup, validated on 9+ buildings with LOBO cross‑validation.


---

## 8. SHACL Validation: Formal Constraints as an Explainability Layer

### 8.1 Why SHACL?

**SHACL** (Shapes Constraint Language) is a W3C standard for expressing and checking constraints over RDF graphs. In our context it provides a **declarative, standards‑based** way to define BIM rules, instead of scattering ad‑hoc checks across scripts.

| Approach               | Pros                                 | Cons                              |
|------------------------|--------------------------------------|-----------------------------------|
| Hardcoded Python rules | Quick to implement                   | Hard to reuse, hard to audit      |
| Cypher (Neo4j)         | Natural for property graphs          | Database‑specific, not a web standard |
| **SHACL**              | Standardised, declarative, good tool support | Requires RDF representation |

SHACL is especially useful when we want BIM checks to be **transparent, reviewable and shareable** between tools.

### 8.2 From IFC Graph to RDF

To use SHACL, we map our IFC‑based graph into a simple RDF representation (Turtle):

- Each graph node becomes an RDF resource, with `rdf:type` set to the IFC class  
  (e.g. `bim:IfcWallStandardCase`, `bim:IfcDoor`).
- Each graph edge becomes an RDF triple  
  (e.g. `bim:IfcSpace_1 bim:boundedBy bim:IfcWallStandardCase_5`).
- Selected properties become datatype properties  
  (e.g. `bim:hasFireRating "60"^^xsd:integer`).

**Lightweight ontology vs full ifcOWL.**  
The official ifcOWL ontology is large and complex. For this notebook we use a **minimal BIM vocabulary** that only includes the classes and predicates we actually need. This keeps both the RDF and the SHACL shapes compact and readable.

### 8.3 Alternative Path: Cypher Checks in Neo4j

The same integrity rules can also be expressed as **Cypher** queries and run directly on a Neo4j graph. This is practical when:

- the BIM graph is already stored in Neo4j,
- checks should run in CI/CD or dashboards,

In this model:

- Node labels correspond to IFC classes (e.g. `:IfcDoor`, `:IfcSpace`, `:IfcOpeningElement`),
- Relationship types correspond to semantic predicates (e.g. `:filledBy`, `:hasOpening`, `:boundedBy`).

In this notebook SHACL is treated as the **primary, standards‑based** constraint layer, while Cypher is shown as a **practical implementation option** for graph databases.


In [19]:
CYPHER_CHECKS = {
    "doors_without_exactly_one_opening": """
MATCH (d:IfcDoor)
OPTIONAL MATCH (o:IfcOpeningElement)-[:filledBy]->(d)
WITH d, count(o) AS c
WHERE c <> 1
RETURN d.ifcId AS ifcId, c AS opening_count
ORDER BY ifcId
""",
    "openings_without_host": """
MATCH (o:IfcOpeningElement)
WHERE NOT (()-[:hasOpening]->(o))
RETURN o.ifcId AS ifcId
ORDER BY ifcId
""",
    "spaces_without_boundaries": """
MATCH (s:IfcSpace)
WHERE NOT (s)-[:boundedBy]->()
RETURN s.ifcId AS ifcId
ORDER BY ifcId
""",
}

for name, query in CYPHER_CHECKS.items():
    print(f"\n=== {name} ===")
    print(query.strip())


=== doors_without_exactly_one_opening ===
MATCH (d:IfcDoor)
OPTIONAL MATCH (o:IfcOpeningElement)-[:filledBy]->(d)
WITH d, count(o) AS c
WHERE c <> 1
RETURN d.ifcId AS ifcId, c AS opening_count
ORDER BY ifcId

=== openings_without_host ===
MATCH (o:IfcOpeningElement)
WHERE NOT (()-[:hasOpening]->(o))
RETURN o.ifcId AS ifcId
ORDER BY ifcId

=== spaces_without_boundaries ===
MATCH (s:IfcSpace)
WHERE NOT (s)-[:boundedBy]->()
RETURN s.ifcId AS ifcId
ORDER BY ifcId


In [20]:
from rdflib import Graph as RDFGraph, Namespace, Literal, RDF, RDFS, XSD

BIM = Namespace("http://example.org/bim#")
g_rdf = RDFGraph()
g_rdf.bind("bim", BIM)
g_rdf.bind("xsd", XSD)

id_to_uri = {}
node_type_col = "ifc_type" if "ifc_type" in nodes_df.columns else "node_type"

for _, row in nodes_df.iterrows():
    ifc_id = int(row["ifc_id"])
    ifc_type = str(row.get(node_type_col, "UnknownIfcType"))

    uri = BIM[f"{ifc_type}_{ifc_id}"]
    id_to_uri[ifc_id] = uri

    # Keep original IFC class names as RDF classes, e.g. bim:IfcDoor.
    g_rdf.add((uri, RDF.type, BIM[ifc_type]))
    g_rdf.add((uri, BIM.ifcId, Literal(ifc_id, datatype=XSD.integer)))
    if pd.notna(row.get("name")):
        g_rdf.add((uri, RDFS.label, Literal(str(row["name"]))))

REL_PRED = {
    "IfcRelSpaceBoundary": BIM.boundedBy,
    "IfcRelVoidsElement": BIM.hasOpening,
    "IfcRelFillsElement": BIM.filledBy,
}

edges_for_rdf = edge_rows if "edge_rows" in globals() else edges_df
for _, row in edges_for_rdf.iterrows():
    s, t = int(row["source"]), int(row["target"])
    if s in id_to_uri and t in id_to_uri:
        pred = REL_PRED.get(row["relation"], BIM.relatedTo)
        g_rdf.add((id_to_uri[s], pred, id_to_uri[t]))

print(f"RDF graph: {len(g_rdf)} triples  ({len(id_to_uri)} resources)")
ttl = g_rdf.serialize(format="turtle")
print("\nSample Turtle (first 35 lines):\n")
print("\n".join(ttl.splitlines()[:35]))

RDF graph: 734 triples  (158 resources)

Sample Turtle (first 35 lines):

@prefix bim: <http://example.org/bim#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

bim:IfcSpace_1059 a bim:IfcSpace ;
    rdfs:label "Bathroom 2" ;
    bim:boundedBy bim:IfcCovering_24186,
        bim:IfcDoor_16115,
        bim:IfcDoor_35413,
        bim:IfcSlab_21336,
        bim:IfcSlab_6247,
        bim:IfcWallStandardCase_31431,
        bim:IfcWallStandardCase_35357,
        bim:IfcWallStandardCase_5948,
        bim:IfcWallStandardCase_6080 ;
    bim:ifcId 1059 .

bim:IfcSpace_1218 a bim:IfcSpace ;
    rdfs:label "Bedroom 2" ;
    bim:boundedBy bim:IfcCovering_24544,
        bim:IfcDoor_8386,
        bim:IfcSlab_20958,
        bim:IfcSlab_6247,
        bim:IfcWallStandardCase_5498,
        bim:IfcWallStandardCase_5548,
        bim:IfcWallStandardCase_5903,
        bim:IfcWallStandardCase_5948,
        bim:IfcWallStandardCase_6080,
        bim:I

### 8.3 Defining SHACL Shapes

SHACL “shapes” capture domain rules for the **original IFC classes** in a declarative way.

**Example shapes:**

1. **DoorFillsOpeningShape**  
   Every `bim:IfcDoor` must be linked to exactly one opening  
   via the inverse property `bim:filledBy`.

2. **OpeningInHostShape**  
   Every `bim:IfcOpeningElement` must be hosted by at least one element  
   via the inverse property `bim:hasOpening`.

3. **SpaceBoundaryShape**  
   Every `bim:IfcSpace` must be bounded by at least one building element.

These shapes are useful in two ways:

- **Data quality:** they detect missing or inconsistent relations/properties in the original IFC model.
- **Prediction audit:** after the GNN predicts properties, we can check if its outputs still satisfy the SHACL constraints.


In [21]:
import pyshacl

SHACL_SHAPES = """
@prefix sh:   <http://www.w3.org/ns/shacl#> .
@prefix bim:  <http://example.org/bim#> .
@prefix xsd:  <http://www.w3.org/2001/XMLSchema#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

bim:DoorFillsOpeningShape
    a sh:NodeShape ;
    sh:targetClass bim:IfcDoor ;
    sh:property [
        sh:path [ sh:inversePath bim:filledBy ] ;
        sh:minCount 1 ;
        sh:maxCount 1 ;
        sh:message "IfcDoor must be linked to exactly one IfcOpeningElement via filledBy" ;
    ] .

bim:OpeningInHostShape
    a sh:NodeShape ;
    sh:targetClass bim:IfcOpeningElement ;
    sh:property [
        sh:path [ sh:inversePath bim:hasOpening ] ;
        sh:minCount 1 ;
        sh:message "IfcOpeningElement must be hosted by at least one element via hasOpening" ;
    ] .

bim:SpaceBoundaryShape
    a sh:NodeShape ;
    sh:targetClass bim:IfcSpace ;
    sh:property [
        sh:path bim:boundedBy ;
        sh:minCount 1 ;
        sh:message "IfcSpace must be bounded by at least one element" ;
    ] .

bim:LabelShape
    a sh:NodeShape ;
    sh:targetClass bim:IfcWall, bim:IfcWallStandardCase, bim:IfcDoor, bim:IfcWindow, bim:IfcSlab, bim:IfcSpace ;
    sh:property [
        sh:path rdfs:label ;
        sh:minCount 1 ;
        sh:message "Element must have a human-readable label" ;
    ] .
"""

conforms, results_graph, results_text = pyshacl.validate(
    data_graph=g_rdf,
    shacl_graph=SHACL_SHAPES,
    data_graph_format="json-ld",
    shacl_graph_format="turtle",
    inference="none",
    abort_on_first=False,
)

print(f"SHACL validation conforms: {conforms}\n")
if not conforms:
    for line in results_text.strip().splitlines():
        print(line)
else:
    print("All shapes passed.")

SHACL validation conforms: True

All shapes passed.


### 8.4 SHACL as an Explainability Layer

In a production co‑pilot, GNN predictions would be written back into the RDF graph and then checked again with SHACL:

- Predictions that **pass** all SHACL shapes are consistent with the encoded domain rules → they can be treated as higher‑confidence suggestions.
- Predictions that **violate** a shape are **flagged for review**, and SHACL reports exactly **which constraint** was broken.

This links **learned patterns** (GNN) with **explicit rules** (SHACL), so the architect sees both: a data‑driven suggestion and a clear rule‑based explanation of potential issues.


In [22]:
g_rdf_test = RDFGraph()
for s, p, o in g_rdf:
    g_rdf_test.add((s, p, o))
g_rdf_test.bind("bim", BIM)

door_uris = [s for s, p, o in g_rdf_test.triples((None, RDF.type, BIM.IfcDoor))]
if door_uris:
    victim = door_uris[0]
    removed = list(g_rdf_test.triples((None, BIM.filledBy, victim)))
    for triple in removed:
        g_rdf_test.remove(triple)
    print(f"Removed {len(removed)} filledBy edge(s) targeting {victim.split('#')[-1]}")

    conforms2, _, text2 = pyshacl.validate(
        data_graph=g_rdf_test,
        shacl_graph=SHACL_SHAPES,
        data_graph_format="json-ld",
        shacl_graph_format="turtle",
        inference="none",
        abort_on_first=False,
    )
    print(f"\nAfter tampering  conforms: {conforms2}\n")
    for line in text2.strip().splitlines():
        if "IfcDoor" in line or "filledBy" in line or "Violation" in line or "Result" in line:
            print(line)
else:
    print("No IfcDoor in RDF graph.")

print("\n--- Co-pilot workflow: GNN predicts -> SHACL audits -> violations flagged ---")

Removed 1 filledBy edge(s) targeting IfcDoor_8169

After tampering  conforms: False

Results (1):
Constraint Violation in MinCountConstraintComponent (http://www.w3.org/ns/shacl#MinCountConstraintComponent):
	Severity: sh:Violation
	Source Shape: [ sh:maxCount Literal("1", datatype=xsd:integer) ; sh:message Literal("IfcDoor must be linked to exactly one IfcOpeningElement via filledBy") ; sh:minCount Literal("1", datatype=xsd:integer) ; sh:path [ sh:inversePath bim:filledBy ] ]
	Focus Node: bim:IfcDoor_8169
	Result Path: [ sh:inversePath bim:filledBy ]
	Message: IfcDoor must be linked to exactly one IfcOpeningElement via filledBy

--- Co-pilot workflow: GNN predicts -> SHACL audits -> violations flagged ---


---

## 9. Pipeline Overview Diagram

A visual summary of the full pipeline, from IFC file to co-pilot output.

### End-to-End Pipeline: IFC to Co-Pilot

```mermaid
flowchart LR
    A[IFC File] --> B[Geometry Extraction<br/>Sec 3]
    B --> C[Point Cloud Sampling<br/>Sec 4]
    C --> D[HeteroData Graph<br/>Sec 5]
    D --> E[Feature Engineering<br/>Sec 6]
    E --> F[R-GCN<br/>Sec 7]
    F --> G[RDF Conversion<br/>Sec 8]
    G --> H[SHACL Validation<br/>Sec 8]
    H --> I[Co-Pilot Output]
```

IFC File -> Geometry Extraction -> Point Cloud Sampling -> HeteroData Graph -> Feature Engineering -> R-GCN -> RDF Conversion -> SHACL Validation -> Co-Pilot Output

---

## 10. Discussion & Trade-offs

### 10.1 Why a Graph-Centred Representation?

In this notebook the **graph** is the primary representation; meshes and point clouds are used to derive features but are not the core data structure.

This choice is driven by:

- The **relational nature** of many BIM tasks: whether a door needs a fire rating depends on which space it connects, which wall it fills, where the escape routes are, etc. This is naturally expressed as a graph.
- The **native structure of IFC**: relations such as `IfcRelSpaceBoundary`, `IfcRelVoidsElement`, `IfcRelFillsElement` already encode design intent and fit directly into graph edges.
- **Scalability**: the graph has one node per element/space, not per mesh vertex or voxel, which keeps models lightweight.

**Limitation.**  
A pure graph discards very fine geometric detail. Tasks that need millimetre‑level reasoning about surfaces (e.g. crack detection, very fine clash detection) would require mesh‑ or point‑cloud‑based models, or tighter coupling to a geometric encoder.

---

### 10.2 HeteroData vs Homogeneous Graph with Edge Types

In this notebook, `HeteroData` is used mainly to **make IFC semantics explicit** (separate node and edge types). This is helpful for explanation and for showing how the representation aligns with the schema.

For a production pipeline, a **homogeneous graph with relation types + R‑GCN** is often the more practical choice:

- all nodes share one feature space (with `ifc_type` encoded as a feature),
- relation types are handled by R‑GCN via separate weights per edge type,
- the number of parameters stays manageable even on small BIM datasets.

A possible hybrid strategy is:

- keep major IFC classes as separate types,
- group rare types into a generic “other” category,
- and choose between `HeteroData` and homogeneous R‑GCN depending on dataset size and task.

---

### 10.3 SHACL vs Learned Rules

SHACL and GNNs play complementary roles:

- **SHACL constraints** are explicit, crisp and auditable. They express “hard” rules (e.g. “every door on an escape route must have a fire rating”) and are easy to review and share.
- **GNN predictions** are learned from data and can capture subtle patterns that would be hard to write as rules, but they are inherently probabilistic and less transparent.

Combining them:

- the GNN proposes values or flags anomalies based on learned patterns,
- SHACL then checks whether the resulting graph still satisfies key constraints,
- violations are highlighted together with the exact rule that was broken.

This combination is well suited for a **co‑pilot**: it preserves a clear rule layer while still benefiting from data‑driven suggestions.

---

### 10.4 Limitations & Future Directions

1. **Single‑building demo.**  
   The notebook works on one sample building (Duplex.ifc) and therefore does not include GNN training. Real evaluation requires many buildings and a protocol such as Leave‑One‑Building‑Out, as used in the master’s pipeline.

2. **Limited labels.**  
   Supervised node classification depends on having enough labelled elements. For typical BIM data, semi‑supervised or self‑supervised approaches (graph autoencoders, contrastive learning, pretraining à la BIGNet) are a promising direction.

3. **Schema variability.**  
   The current pipeline targets a specific subset of IFC classes and relations. A production system must handle differences between IFC2x3 / IFC4 / IFC4.3 and project‑specific modelling practices.

4. **RDF/ontology scope.**  
   The RDF/SHACL layer here uses a minimal BIM vocabulary tuned to this notebook. A full integration with ifcOWL and Linked Building Data (LBD) ontologies would enable interoperability with broader Semantic Web tooling.

5. **Richer geometric features.**  
   OBB‑based geometry and simple shape descriptors are a good baseline. For tasks where shape is critical, it would be natural to add mesh‑level features (normals, curvature) or even a learned point‑cloud encoder (e.g. PointNet) as an additional feature block feeding into the GNN.

Overall, the notebook focuses on **clear, reusable representations** (graph + multi‑level geometry + SHACL constraints) that match the co‑pilot framing of the ICD/CA assignment, while the heavier training and evaluation setup is delegated to the multi‑building master’s pipeline.


## References

1. Pauwels, P., & Terkaj, W. (2016). *Express to OWL for construction industry: Towards a recommendable and usable ifcOWL ontology.* Automation in Construction, 63, 100–133.
2. Schlichtkrull, M., et al. (2018). *Modeling Relational Data with Graph Convolutional Networks.* ESWC 2018.
3. Hu, Z., et al. (2020). *Heterogeneous Graph Transformer.* WWW 2020.
4. Knublauch, H., & Kontokostas, D. (2017). *Shapes Constraint Language (SHACL).* W3C Recommendation.
5. Qi, C. R., et al. (2017). *PointNet: Deep Learning on Point Sets for 3D Classification and Segmentation.* CVPR 2017.
6. Fey, M., & Lenssen, J. E. (2019). *Fast Graph Representation Learning with PyTorch Geometric.* ICLR Workshop on Representation Learning on Graphs and Manifolds.